# Extração de features por CPE

- Set A (principal): 24 features homogéneas - percentagem de consumo diário por hora. Todas na mesma escala (%), naturalmente independentes e não requerem normalização adicional.
- Set B (complementar): features derivadas que capturam variabilidade, tendência e contexto operacional. Para comparação posterior. 

A análise de clustering será feita primeiro com o Set A, e depois repetida
com o Set B para comparar a qualidade percebida dos agrupamentos.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

base_dir = Path("../results")

# Pastas principais
exploration_dir = base_dir / "exploration"
features_dir = base_dir / "features"
clustering_dir = base_dir / "clustering"
anomalies_dir = base_dir / "anomalies"
models_dir = base_dir / "models"

# Subpastas específicas
intermediate_dir = exploration_dir / "intermediate"

# Clustering
clustering_analysis_dir = clustering_dir / "analysis"
clustering_plots_dir = clustering_dir / "plots"

# Features
features_plots_dir = features_dir / "plots"
features_validation_dir = features_dir / "validation"

# Anomalies
anomalies_plots_dir = anomalies_dir / "plots"
anomalies_temporal_plots_dir = anomalies_dir / "anomalies_temporal_plots"

for d in [
    exploration_dir,
    features_dir,
    clustering_dir,
    anomalies_dir,
    models_dir,
    intermediate_dir,
    clustering_analysis_dir,
    clustering_plots_dir,
    features_plots_dir,
    features_validation_dir,
    anomalies_plots_dir,
    anomalies_temporal_plots_dir,
]:
    d.mkdir(parents=True, exist_ok=True)

df_pivot = pd.read_csv(intermediate_dir / "df_pivot.csv", index_col=0, parse_dates=True)
perfil_horario = pd.read_csv(intermediate_dir / "perfil_horario.csv", index_col=0)
perfil_utilizacao = pd.read_csv(intermediate_dir / "perfil_utilizacao.csv")
consumo_diario_cpe = pd.read_csv(intermediate_dir / "consumo_diario_cpe.csv")
perfil_weekday = pd.read_csv(intermediate_dir / "perfil_weekday.csv", index_col=0)
perfil_weekend = pd.read_csv(intermediate_dir / "perfil_weekend.csv", index_col=0)
consumo_diario = pd.read_csv(intermediate_dir / "consumo_diario.csv", index_col=0, parse_dates=True)
consumo_mensal = pd.read_csv(intermediate_dir / "consumo_mensal.csv", index_col=0, parse_dates=True)

perfil_horario.index = perfil_horario.index.astype(int)
perfil_weekday.index = perfil_weekday.index.astype(int)
perfil_weekend.index = perfil_weekend.index.astype(int)

print("Ficheiros intermédios carregados com sucesso.")
display(perfil_horario.head())

In [ ]:
# SET A — Perfil horário normalizado (24 features)
# Cada feature representa a percentagem do consumo diário total que ocorre numa determinada hora do dia.
# Exemplo: se f10 = 12.5%, significa que em média 12.5% do consumo diário deste CPE acontece entre as 10:00 e as 11:00.

# Normalizar: cada hora como % do total diário
perfil_horario_pct = perfil_horario.copy()
totais_diarios = perfil_horario_pct.sum(axis=0)  # soma das 24 horas por CPE
 
# Evitar divisão por zero (CPEs com consumo total ~0)
totais_diarios = totais_diarios.replace(0, np.nan)
 
perfil_horario_pct = (perfil_horario_pct / totais_diarios) * 100
 
# Transpor: cada linha = 1 CPE, cada coluna = 1 hora
features_setA = perfil_horario_pct.T.copy()
features_setA.columns = [f"f{h+1:02d}_pct_hora_{h:02d}" for h in range(24)]
features_setA.index.name = "CPE"

In [ ]:
# Verificação

print("SET A — Perfil horário normalizado (24 features)")

print(f"Shape: {features_setA.shape}")
print(f"Soma por CPE (deve ser ~100%):")
somas = features_setA.sum(axis=1)
print(f"  Min: {somas.min():.2f}%")
print(f"  Max: {somas.max():.2f}%")
print(f"  Mean: {somas.mean():.2f}%")

In [ ]:
# Remover CPEs com consumo total ~0 (se existirem)

mask_valido = ~features_setA.isna().any(axis=1)
n_removidos = (~mask_valido).sum()
if n_removidos > 0:
    print(f"\nRemovidos {n_removidos} CPEs com consumo total ~0")
    features_setA = features_setA[mask_valido]
 
print(f"\nShape final: {features_setA.shape}")
display(features_setA.head())

In [ ]:
# Visualização: perfil médio global

plt.figure(figsize=(10, 4))
features_setA.mean().plot(kind="bar", color="#0D7C66", alpha=0.8)
plt.title("Perfil médio horário (% do consumo diário)")
plt.xlabel("Hora do dia")
plt.ylabel("% do consumo diário")
plt.xticks(range(24), [f"{h}h" for h in range(24)], rotation=45)
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(features_plots_dir / "perfil_horario_pct_medio.png", dpi=150)
plt.show()

In [ ]:
# Visualização: heatmap de todos os CPEs

plt.figure(figsize=(14, 8))
sns.heatmap(
    features_setA.sort_values("f01_pct_hora_00", ascending=False),
    cmap="YlOrRd",
    xticklabels=[f"{h}h" for h in range(24)],
    yticklabels=False,
    cbar_kws={"label": "% consumo diário"}
)
plt.title("Distribuição horária do consumo por CPE (Set A)")
plt.xlabel("Hora do dia")
plt.ylabel("CPE (ordenados por consumo nocturno)")
plt.tight_layout()
plt.savefig(features_plots_dir / "heatmap_perfil_horario_pct.png", dpi=150)
plt.show()

In [ ]:
# Guardar

features_setA.to_csv(features_dir / "features_setA.csv")
print(f"\nSet A guardado: {features_dir / 'features_setA.csv'}")

In [ ]:
# Estatísticas descritivas

print("\nEstatísticas descritivas do Set A:")
display(features_setA.describe().T.round(2))

In [ ]:
# SET B — Features derivadas (complementar, para comparação)

# Estas features capturam dimensões diferentes: intensidade absoluta,
# variabilidade, regularidade, tendência e contexto operacional.
# Serão usadas numa segunda iteração de clustering para comparação.
 
from scipy import stats
 
features_setB = pd.DataFrame(index=df_pivot.columns)
features_setB.index.name = "CPE"
 
# Estatísticas básicas
features_setB["mean"] = df_pivot.mean()
features_setB["std"] = df_pivot.std()
features_setB["max"] = df_pivot.max()
features_setB["median"] = df_pivot.median()
features_setB["p05"] = df_pivot.quantile(0.05)
features_setB["p95"] = df_pivot.quantile(0.95)
features_setB["iqr"] = df_pivot.quantile(0.75) - df_pivot.quantile(0.25)
 
# Qualidade
features_setB["missing_pct"] = df_pivot.isna().mean()
features_setB["pct_zero"] = (df_pivot < 0.01).mean()
 
# Padrões temporais
features_setB["hora_pico"] = perfil_horario.idxmax()
mean_noite = perfil_horario.loc[0:6].mean()
mean_dia = perfil_horario.loc[7:20].mean()
features_setB["mean_noite"] = mean_noite
features_setB["mean_dia"] = mean_dia
features_setB["ratio_noite_dia"] = np.where(
    features_setB["mean_dia"] > 1e-6,
    features_setB["mean_noite"] / features_setB["mean_dia"],
    np.nan
)
features_setB["amplitude_horaria"] = perfil_horario.max() - perfil_horario.min()
 
# Comportamento operacional
perfil_utilizacao_wide = (
    perfil_utilizacao
    .pivot_table(index=["CPE", "hora"], columns="tipo_utilizacao", values="PotActiva")
    .reset_index()
)
diff_utilizacao = (
    perfil_utilizacao_wide
    .assign(diff=lambda x: (x["utilizacao"] - x["nao_utilizacao"]).abs())
    .groupby("CPE")["diff"]
    .mean()
)
features_setB["diff_utilizacao_nao_utilizacao"] = diff_utilizacao
 
cpe_com_nao_util = (
    consumo_diario_cpe
    .groupby("CPE")["tipo_utilizacao"]
    .apply(lambda x: (x == "nao_utilizacao").any())
)
features_setB["tem_dias_nao_utilizacao"] = cpe_com_nao_util.astype(int)
 
hora_pico_util = (
    perfil_utilizacao[perfil_utilizacao["tipo_utilizacao"] == "utilizacao"]
    .sort_values(["CPE", "PotActiva"], ascending=[True, False])
    .drop_duplicates("CPE")
    .set_index("CPE")["hora"]
)
hora_pico_nao_util = (
    perfil_utilizacao[perfil_utilizacao["tipo_utilizacao"] == "nao_utilizacao"]
    .sort_values(["CPE", "PotActiva"], ascending=[True, False])
    .drop_duplicates("CPE")
    .set_index("CPE")["hora"]
)
features_setB["hora_pico_utilizacao"] = hora_pico_util
features_setB["hora_pico_nao_utilizacao"] = hora_pico_nao_util
 
diff_weekday_weekend = (perfil_weekday - perfil_weekend).abs().mean()
features_setB["diff_weekday_weekend"] = diff_weekday_weekend
 
# Variabilidade e regularidade
features_setB["cv"] = np.where(
    features_setB["mean"] > 1e-6,
    features_setB["std"] / features_setB["mean"],
    np.nan
)
features_setB["cv_diario"] = np.where(
    consumo_diario.mean() > 1e-6,
    consumo_diario.std() / consumo_diario.mean(),
    np.nan
)
 
autocorr_lag1 = {}
autocorr_lag96 = {}
for cpe in df_pivot.columns:
    serie = df_pivot[cpe].dropna()
    autocorr_lag1[cpe] = serie.autocorr(lag=1) if len(serie) > 2 else np.nan
    autocorr_lag96[cpe] = serie.autocorr(lag=96) if len(serie) > 96 else np.nan
features_setB["autocorr_lag1"] = pd.Series(autocorr_lag1)
features_setB["autocorr_lag96"] = pd.Series(autocorr_lag96)
 
# Tendência e sazonalidade
slope_por_cpe = {}
for cpe in consumo_diario.columns:
    serie = consumo_diario[cpe].dropna()
    if len(serie) > 10:
        x = np.arange(len(serie))
        result = stats.linregress(x, serie.values)
        slope_por_cpe[cpe] = result.slope
    else:
        slope_por_cpe[cpe] = np.nan
features_setB["tendencia_slope"] = pd.Series(slope_por_cpe)
features_setB["tendencia_norm"] = np.where(
    features_setB["mean"] > 1e-6,
    features_setB["tendencia_slope"] / consumo_diario.mean(),
    np.nan
)
features_setB["cv_mensal"] = np.where(
    consumo_mensal.mean() > 1e-6,
    consumo_mensal.std() / consumo_mensal.mean(),
    np.nan
)

In [ ]:
# Limpeza

cols_para_imputar = ["diff_utilizacao_nao_utilizacao", "hora_pico_utilizacao", "hora_pico_nao_utilizacao"]
for col in cols_para_imputar:
    if col in features_setB.columns:
        features_setB[col] = features_setB[col].fillna(features_setB[col].median())
 
features_setB = features_setB.dropna()
 
print("SET B — Features derivadas (26 features)")

print(f"Shape: {features_setB.shape}")
 
categorias = {
    "Estatísticas básicas": ["mean", "std", "max", "median", "p05", "p95", "iqr"],
    "Qualidade": ["missing_pct", "pct_zero"],
    "Padrões temporais": ["hora_pico", "mean_noite", "mean_dia", "ratio_noite_dia", "amplitude_horaria"],
    "Comportamento operacional": ["diff_utilizacao_nao_utilizacao", "tem_dias_nao_utilizacao",
                                   "hora_pico_utilizacao", "hora_pico_nao_utilizacao", "diff_weekday_weekend"],
    "Variabilidade e regularidade": ["cv", "cv_diario", "autocorr_lag1", "autocorr_lag96"],
    "Tendência e sazonalidade": ["tendencia_slope", "tendencia_norm", "cv_mensal"],
}
for cat, cols in categorias.items():
    cols_presentes = [c for c in cols if c in features_setB.columns]
    print(f"  {cat}: {len(cols_presentes)} features")

In [ ]:
# Guardar

features_setB.to_csv(features_dir / "features_setB.csv")
print(f"\nSet B guardado: {features_dir / 'features_setB.csv'}")

In [ ]:
# Para retrocompatibilidade com notebooks seguintes

features_setA.to_csv(features_dir / "features_cpe.csv")
print(f"Set A copiado para features_cpe.csv (usado pelo clustering)")

In [ ]:
# Resumo comparativo

print("RESUMO")

print(f"\n  Set A (perfil horário):  {features_setA.shape[1]} features · {features_setA.shape[0]} CPEs")
print(f"  Set B (derivadas):      {features_setB.shape[1]} features · {features_setB.shape[0]} CPEs")
print(f"\n  → Clustering começa com Set A")
print(f"  → Depois repete-se com Set B para comparação")

# Validação e análise das features extraídas

Antes de avançar para o clustering, é importante validar que as features são informativas e não redundantes.

In [ ]:
# Correlação entre features - Set A

plt.figure(figsize=(14, 10))

corr_A = features_setA.corr()
mask = np.triu(np.ones_like(corr_A, dtype=bool))

sns.heatmap(
    corr_A,
    mask=mask,
    annot=True,
    fmt=".1f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    annot_kws={"size": 7}
)

plt.title("Matriz de correlação das features — Set A")
plt.tight_layout()
plt.savefig(features_validation_dir / "correlacao_features_setA.png", dpi=150)
plt.show()

# Identificar pares altamente correlacionados (|r| > 0.9)
high_corr_pairs_A = []

for i in range(len(corr_A.columns)):
    for j in range(i + 1, len(corr_A.columns)):
        if abs(corr_A.iloc[i, j]) > 0.9:
            high_corr_pairs_A.append({
                "feature_1": corr_A.columns[i],
                "feature_2": corr_A.columns[j],
                "correlacao": corr_A.iloc[i, j]
            })

if high_corr_pairs_A:
    print("Set A — Pares de features com correlação > 0.9:")
    display(
        pd.DataFrame(high_corr_pairs_A).sort_values(
            "correlacao", key=abs, ascending=False
        )
    )
    print("\nNota: features altamente correlacionadas podem ser redundantes.")
    print("Considerar remoção de uma de cada par antes do clustering.")
else:
    print("Set A — Nenhum par de features com correlação > 0.9 encontrado.")

In [ ]:
# Distribuição das features - Set A

fig, axes = plt.subplots(5, 5, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(features_setA.columns):
    if i < len(axes):
        features_setA[col].hist(
            bins=20,
            ax=axes[i],
            color="#0D7C66",
            alpha=0.7,
            edgecolor="white"
        )
        axes[i].set_title(col, fontsize=9, fontweight="bold")
        axes[i].tick_params(labelsize=7)

# Esconder eixos não usados
for j in range(len(features_setA.columns), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuição das features por CPE — Set A", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(features_validation_dir / "distribuicao_features_setA.png", dpi=150)
plt.show()

In [ ]:
# Estatísticas descritivas completas - Set A

print("\nSet A — Estatísticas descritivas:")
display(features_setA.describe().T.round(3))

In [ ]:
# Correlação entre features - Set B

plt.figure(figsize=(16, 12))

corr_B = features_setB.corr()
mask = np.triu(np.ones_like(corr_B, dtype=bool))

sns.heatmap(
    corr_B,
    mask=mask,
    annot=True,
    fmt=".1f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    annot_kws={"size": 6}
)

plt.title("Matriz de correlação das features — Set B")
plt.tight_layout()
plt.savefig(features_validation_dir / "correlacao_features_setB.png", dpi=150)
plt.show()

# Identificar pares altamente correlacionados (|r| > 0.9)
high_corr_pairs_B = []

for i in range(len(corr_B.columns)):
    for j in range(i + 1, len(corr_B.columns)):
        if abs(corr_B.iloc[i, j]) > 0.9:
            high_corr_pairs_B.append({
                "feature_1": corr_B.columns[i],
                "feature_2": corr_B.columns[j],
                "correlacao": corr_B.iloc[i, j]
            })

if high_corr_pairs_B:
    print("Set B — Pares de features com correlação > 0.9:")
    display(
        pd.DataFrame(high_corr_pairs_B).sort_values(
            "correlacao", key=abs, ascending=False
        )
    )
    print("\nNota: features altamente correlacionadas podem ser redundantes.")
    print("Considerar remoção de uma de cada par antes do clustering.")
else:
    print("Set B — Nenhum par de features com correlação > 0.9 encontrado.")

In [ ]:
# Distribuição das features - Set B

n_features_B = len(features_setB.columns)
n_cols = 5
n_rows = int(np.ceil(n_features_B / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows))
axes = axes.flatten()

for i, col in enumerate(features_setB.columns):
    features_setB[col].hist(
        bins=20,
        ax=axes[i],
        color="#0D7C66",
        alpha=0.7,
        edgecolor="white"
    )
    axes[i].set_title(col, fontsize=9, fontweight="bold")
    axes[i].tick_params(labelsize=7)

# Esconder eixos não usados
for j in range(len(features_setB.columns), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuição das features por CPE — Set B", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(features_validation_dir / "distribuicao_features_setB.png", dpi=150)
plt.show()

In [ ]:
# SET B — Estatísticas descritivas

print("\nSet B — Estatísticas descritivas:")
display(features_setB.describe().T.round(3))